In [ ]:
import pandas as pd
import re

HARD_BLOCKS = [
    'how to kill', 'how to make a bomb', 'how to make bomb', 'how to hurt',
    'suicide method', 'self harm method', 'buy illegal', 'sex with', 'sexual',
    'porn', 'nude', 'naked', 'masturbat', 'rape', 'molest', 'pedophil',
    'nigger', 'faggot',
]

NOT_A_CLAIM_PATTERNS = [
    r'^(daily|weekly|monthly)\s+(simple\s+)?questions?\s+thread',
    r'^(moronic\s+monday|rant\s+wednesday|victory\s+sunday|gym\s+story)',
    r'^new\s+to\s+r/', r'^r/\w+\s+weekly', r'^r/\w+\s+\*\*new',
    r'^\[meta\]', r'^ama\s+series', r'^panel\s+of\s+scientists',
    r'^subreddit\s+statement', r'^popular\s+topics\s+mega', r'megathread', r'^mod\s+post',
]
NOT_A_CLAIM_RE = re.compile('|'.join(NOT_A_CLAIM_PATTERNS), re.IGNORECASE)

CLAIM_SIGNALS = [
    r'\bi (think|believe|feel|heard|was told|always thought|assumed|read|noticed|found)\b',
    r'\bi\'ve (always|never|heard|believed|thought|read|seen)\b',
    r'\bi was always told\b', r'\bi always (thought|assumed|believed|heard)\b',
    r'\beveryone (knows|says|thinks|agrees|believes)\b', r'\bmost people\b',
    r'\bcommon (knowledge|sense|belief|myth)\b', r'\bpeople say\b', r'\bpeople think\b',
    r'\bthey say\b', r'\bstudies show\b', r'\bscience says\b',
    r'\bscientifically (proven|shown)\b', r'\bproven\b', r'\bexperts say\b',
    r'\bdoctors say\b', r'\bresearch shows\b',
    r'\bmy (doctor|trainer|mom|dad|friend|teacher|professor|boss) (told|said|thinks)\b',
    r'\bmyth\b', r'\bmisconception\b', r'\bactually\b', r'\bturns out\b',
    r'\bsurprisingly\b', r'\bcontrary to\b', r'\bis it true\b', r'\bisn\'t it true\b',
    r'\bdoes .{3,40} really\b', r'\baren\'t you supposed to\b',
    r'\bisn\'t it (better|worse|true|right|correct)\b',
    r'\byou should (always|never|never ever)\b', r'\byou (always|never) should\b',
    r'\bthe (best|right|correct|only|proper) way to\b', r'\byou can\'t\b',
    r'\byou have to\b', r'\byou need to\b', r'\byou must\b',
    r'\bthe (best|worst|biggest|most|least)\b',
    r'\bis (overrated|underrated|better|worse|the best|the worst)\b',
    r'\bchange my (view|mind)\b', r'\bcmv:\b', r'\bunpopular opinion\b',
    r'\bhot take\b', r'\bfight me\b', r'\bchange my view\b',
    r'\b(healthy|unhealthy|bad for you|good for you|causes|prevents)\b',
    r'\bnatural(ly)?\b', r'\btoxic\b', r'\bdetox\b', r'\bboosting\b',
    r'\bwhy do(es)? (everyone|people|we)\b', r'\bwhy is it (that|when)\b',
    r'\bwhy (don\'t|doesn\'t|can\'t|won\'t)\b',
    r'\bam i (wrong|right|crazy|the only one)\b',
    r'\bis (it|this|that) (normal|weird|bad|wrong|right|okay|true)\b',
]
CLAIM_RE = re.compile('|'.join(CLAIM_SIGNALS), re.IGNORECASE)

HIGH_VALUE_SIGNALS = [
    r'\b(vitamin|supplement|protein|carb|fat|calorie|detox|diet|immune|metabolism)\b',
    r'\b(exercise|workout|muscle|weight loss|cardio|stretching)\b',
    r'\b(sugar|caffeine|alcohol|dairy|gluten|organic|processed)\b',
    r'\b(myth|misconception|actually|debunked|wrong about)\b',
    r'\b(introvert|extrovert|personality|habit|willpower|addiction)\b',
    r'\b(left brain|right brain|iq|intelligence|learning style)\b',
    r'\b(lightning|gravity|evolution|brain|memory|sleep|dream)\b',
    r'\b(invest|stock|crypto|saving|debt|credit score|mortgage)\b',
    r'\b(child|kid|baby|toddler|screen time|discipline|spoil)\b',
    r'\b(overrated|underrated|better than|worse than|greatest|worst)\b',
    r'\bcmv\b',
]
HIGH_VALUE_RE = re.compile('|'.join(HIGH_VALUE_SIGNALS), re.IGNORECASE)

FACTUAL_SUBS = {'askscience', 'explainlikeimfive', 'nostupidquestions'}
OPINION_SUBS = {'unpopularopinion', 'changemyview', 'showerthoughts', 'mildlyinfuriating'}
ADVICE_SUBS  = {'relationship_advice', 'amitheasshole', 'personalfinance',
                'parenting', 'nutrition', 'fitness', 'homeimprovement', 'askdocs', 'lifeprotips'}
FACTUAL_WORDS = ['actually','myth','true','fact','really','proven','science','study','research','brain','body','evolution']
OPINION_WORDS = ['think','believe','feel','opinion','better','worse','overrated','underrated','cmv','unpopular','hot take']

def categorize(title, subreddit):
    sub = subreddit.lower().replace('r/', '')
    if sub in FACTUAL_SUBS: return 'factual'
    if sub in OPINION_SUBS: return 'opinion'
    if sub in ADVICE_SUBS:  return 'advice'
    combined = title.lower()
    if any(w in combined for w in FACTUAL_WORDS): return 'factual'
    if any(w in combined for w in OPINION_WORDS): return 'opinion'
    return 'advice'

def is_usable(title, text):
    combined = (str(title) + ' ' + str(text)[:200]).lower()
    t = str(title).strip()
    for kw in HARD_BLOCKS:
        if kw in combined: return False, f'hard_block:{kw}'
    if NOT_A_CLAIM_RE.search(t): return False, 'meta_post'
    if len(t) < 12: return False, 'too_short'
    if not CLAIM_RE.search(combined): return False, 'no_claim_signal'
    return True, 'ok'

def run(input_path, output_claims, output_rejected):
    raw = pd.read_csv(input_path)
    print(f"Raw posts: {len(raw)}")

    accepted, rejected, seen = [], [], set()

    for _, row in raw.iterrows():
        title     = str(row['title']).strip()
        text      = str(row.get('text', ''))
        subreddit = str(row.get('subreddit', ''))

        usable, reason = is_usable(title, text)
        if not usable:
            rejected.append({'title': title, 'subreddit': subreddit, 'reason': reason})
            continue

        key = title[:60].lower()
        if key in seen:
            rejected.append({'title': title, 'subreddit': subreddit, 'reason': 'duplicate'})
            continue
        seen.add(key)

        accepted.append({
            'id':               len(accepted) + 1,
            'wrong_claim':      title,
            'correct_answer':   '',
            'category':         categorize(title, subreddit),
            'research_value':   'high' if HIGH_VALUE_RE.search(title + ' ' + text[:200]) else 'standard',
            'source_subreddit': subreddit,
        })

    claims_df   = pd.DataFrame(accepted)
    rejected_df = pd.DataFrame(rejected)
    claims_df.to_csv(output_claims, index=False)
    rejected_df.to_csv(output_rejected, index=False)

    print(f"Accepted:  {len(claims_df)}")
    print(f"Rejected:  {len(rejected_df)}")
    print(f"Category breakdown:\n{claims_df['category'].value_counts().to_string()}")
    return claims_df

if __name__ == '__main__':
    run(
        input_path='data/raw_reddit.csv',
        output_claims='data/claims_rule_based.csv',
        output_rejected='data/rejected_rule_based.csv',
    )
